# Hasil Training EADRO — tabel dan gambar

Notebook ini **hanya membaca hasil** yang sudah dibuat oleh `pipeline/run_train.sh` (folder `work/result/`). Tidak ada training di sini, jadi aman dijalankan kapan saja, bahkan saat training sedang berjalan di terminal.

**Cara pakai:** pilih kernel `work/envs/train312/bin/python`, lalu jalankan semua sel (*Run All*). Jalankan ulang setiap kali ada run baru selesai.

Kamus singkat (lengkapnya di `Penjelasan_Pekerjaan_EADRO.md`):
- **Val** = data untuk memilih epoch terbaik. **Test** = ujian akhir. Angka yang dilaporkan adalah **Test pada epoch terpilih**.
- **HR@1** = seberapa sering service biang ditebak di urutan pertama. **F1** = ketepatan mendeteksi ada/tidaknya gangguan.
- Akhiran **(std)** = versi baku. Tanpa akhiran = versi kode asli EADRO. Tebakan acak untuk HR@1 hanya 1/27 ≈ 0,037.

In [ ]:
import sys
from pathlib import Path
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "work" / "result").exists():   # bila notebook dibuka dari subfolder
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "pipeline"))
import baca_hasil as bh

RESULT_DIR = ROOT / "work" / "result"
runs, df, files = bh.make_all(str(RESULT_DIR))      # membuat ulang CSV + semua gambar
print(f"{len(runs)} run ditemukan di {RESULT_DIR}")

## 1. Tabel utama — satu baris per run
- **status**: `selesai`, `berjalan` (log masih bertambah), atau `selesai (uji cepat)` (kurang dari 10 epoch; hanya untuk mengecek alur, jangan dipakai sebagai hasil).
- **epoch_terpilih**: epoch dengan HR@1 Val terbaik. Angka `Test ...` di baris yang sama diambil dari epoch itu.
- **menit_latih**: jumlah waktu semua epoch (tidak termasuk evaluasi). Epoch yang jauh lebih lama dari biasanya (mis. laptop tidur di tengah training) dihitung sebesar epoch normal.
- Tabel yang sama tersimpan di `work/result/ringkasan_hasil.csv` (bisa dibuka di Excel/LibreOffice).

In [ ]:
df.style.format(precision=4, na_rep="-").hide(axis="index")

## 2. Rata-rata antar seed
Satu seed saja tidak cukup untuk membandingkan dua metode, karena hasil bisa bergeser sedikit hanya karena pengacakan. Tabel ini menggabungkan beberapa seed untuk skema yang sama sebagai **rata-rata ± simpangan baku**. Hanya run `selesai` dan bukan uji cepat yang ikut dihitung. Bila hanya ada satu seed per skema, simpangan baku belum bisa dihitung.

In [ ]:
agg = bh.aggregate_table(df)
agg if len(agg) else print("Belum ada run penuh yang selesai (epoch >= 10).")

## 3. Kurva training loss
*Loss* adalah ukuran seberapa salah model pada data latih. **Yang sehat: cenderung turun.** Bila naik terus atau mendatar sejak awal, ada yang salah (mis. learning rate terlalu besar). Satu panel per skema, satu garis per seed.

In [ ]:
display(Image(str(RESULT_DIR / "gambar" / "kurva_loss.png")))

## 4. Hasil Val di tiap evaluasi
Evaluasi dilakukan pada epoch 9, 19, 29, 39, 49. **Titik besar = epoch yang dipilih** (Val tertinggi), dan Test dihitung pada epoch itu. Bila garis masih naik di epoch terakhir, model mungkin masih bisa belajar lebih lama. Bila turun setelah puncak, itu tanda mulai menghafal data latih (*overfitting*).

In [ ]:
p = RESULT_DIR / "gambar" / "val_hr1_std.png"
display(Image(str(p))) if p.exists() else print("Belum ada evaluasi Val (butuh minimal sampai epoch 9, atau run 1 epoch).")

## 5. Hasil akhir satu run: versi baku vs versi kode asli
Menampilkan metrik **Test** pada epoch terpilih. Batang biru = versi baku (std), abu-abu = versi kode asli. Pada HR@k, versi asli biasanya lebih rendah karena penyebutnya ikut menghitung jendela normal (lihat Bagian 6 dokumen penjelasan). **Pakai kedua versi secara konsisten saat membandingkan varian fusion.**

Secara bawaan ditampilkan run penuh yang selesai paling baru. Untuk run lain, isi `HASH` dengan nama folder run (kolom `run` pada tabel).

In [ ]:
HASH = None   # contoh: HASH = "a1b2c3d4"
run = bh.pick_run(runs, HASH)
if run is None:
    print("Belum ada run selesai.")
else:
    display(Image(bh.plot_final(run, str(RESULT_DIR / "gambar" / f"hasil_akhir_{run['hash']}.png"))))

## 6. Gambar tersimpan di
`work/result/gambar/` — siap ditempel ke laporan atau slide.

In [ ]:
for f in sorted((RESULT_DIR / "gambar").glob("*.png")):
    print(f.relative_to(ROOT))